![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 20 -- Lab 2: Build a Voice Chatbot

**Scenario:** You are building a voice assistant that can **listen** to you speak, **understand** what you said, **think** of a response, and **speak** it back -- all running locally on this Colab GPU, no API keys needed.

The pipeline chains three models together:

```
Your voice  -->  [Whisper]  -->  text  -->  [Gemma]  -->  response text  -->  [Edge TTS]  -->  audio playback
  (mic)         (speech-to-text)            (LLM)                            (text-to-speech)
```

This lab is mostly **given** code -- your job is to run each part, understand how the pieces connect, and then customize the chatbot with your own personalities.

| Part | Goal |
|---|---|
| Part 1 | Setup: install libraries and load models |
| Part 2 | Whisper: speech to text |
| Part 3 | Microphone: record audio in Colab |
| Part 4 | Gemma: the LLM brain |
| Part 5 | Edge TTS: text to speech |
| Part 6 | Full pipeline: voice in, voice out |
| Part 7 | Personalities: customize your chatbot |
| Part 8 | (Bonus) Arabic voice chatbot |

In [ ]:
!pip install openai-whisper transformers accelerate edge-tts nest-asyncio -q
!apt-get install -y ffmpeg > /dev/null 2>&1

In [ ]:
import torch
import whisper
import edge_tts
import asyncio
import nest_asyncio
import base64
import os

import IPython.display as ipd
from IPython.display import display, Audio, HTML
from google.colab import output as colab_output

nest_asyncio.apply()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
# Part 1 -- Whisper: Speech to Text (GIVEN)

**Whisper** (OpenAI, 2022) converts speech audio into text. It was trained on 680,000 hours of audio in 99 languages. We load the `base` model (~140MB) which fits easily alongside Gemma on a T4 GPU.

In [ ]:
# --- GIVEN: Load Whisper ---
whisper_model = whisper.load_model("base").to(device)
print(f"Whisper 'base' loaded ({sum(p.numel() for p in whisper_model.parameters()):,} parameters)")

In [ ]:
# --- GIVEN: Test Whisper with a sample audio ---
# Download a sample English audio clip
import urllib.request

sample_url = "https://upload.wikimedia.org/wikipedia/commons/e/e9/En-au-hello.ogg"
req = urllib.request.Request(sample_url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as resp, open("sample_hello.ogg", "wb") as f:
    f.write(resp.read())

print("Sample audio:")
display(Audio("sample_hello.ogg"))

result = whisper_model.transcribe("sample_hello.ogg")
print(f"\nTranscription: {result['text']}")
print(f"Detected language: {result['language']}")

In [ ]:
# --- GIVEN: Whisper can also detect language ---
audio = whisper.load_audio("sample_hello.ogg")
audio = whisper.pad_or_trim(audio)
mel = whisper.log_mel_spectrogram(audio).to(device)

_, probs = whisper_model.detect_language(mel)
top_5 = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:5]

print("Top 5 detected languages:")
for lang, prob in top_5:
    print(f"  {lang}: {prob:.1%}")

---
# Part 2 -- Microphone Recording in Colab (GIVEN)

Google Colab runs in a browser, so we use JavaScript to access your microphone. The helper below records for a set number of seconds -- just run the cell, allow mic access when prompted, and speak. The audio is sent back to Python and converted to WAV.

In [ ]:
# --- GIVEN: Microphone recording helper ---

RECORD_JS = """
(async function() {
  try {
    const stream = await navigator.mediaDevices.getUserMedia({audio: true});
    const recorder = new MediaRecorder(stream);
    const chunks = [];
    recorder.ondataavailable = e => chunks.push(e.data);

    return new Promise(resolve => {
      recorder.onstop = () => {
        stream.getTracks().forEach(t => t.stop());
        const blob = new Blob(chunks);
        const reader = new FileReader();
        reader.onloadend = () => resolve(reader.result);
        reader.readAsDataURL(blob);
      };
      recorder.start();
      setTimeout(() => recorder.stop(), %d);
    });
  } catch(e) {
    return "ERROR:" + e.message;
  }
})()
"""

def record_audio(filename="recording.wav", seconds=5):
    """Record audio from microphone for the given number of seconds."""
    print(f"🎤 Recording for {seconds} seconds... speak now!")
    print("   (If prompted, allow microphone access in your browser)")

    data_url = colab_output.eval_js(RECORD_JS % (seconds * 1000))

    if not data_url or str(data_url).startswith("ERROR:"):
        msg = str(data_url).replace("ERROR:", "") if data_url else "Unknown"
        print(f"\n❌ Microphone error: {msg}")
        print("   Fix: click the lock icon in your browser's address bar → allow Microphone → re-run the cell.")
        return None

    header, encoded = data_url.split(",", 1)
    audio_bytes = base64.b64decode(encoded)

    webm_file = filename.replace(".wav", ".webm")
    with open(webm_file, "wb") as f:
        f.write(audio_bytes)

    os.system(f"ffmpeg -y -i {webm_file} -ar 16000 -ac 1 {filename} -loglevel quiet")
    if os.path.exists(webm_file):
        os.remove(webm_file)

    print(f"✓ Saved to {filename}")
    return filename

print("Microphone recorder ready.")

In [ ]:
# --- GIVEN: Test the microphone ---
# Records for 5 seconds, then automatically stops
audio_file = record_audio("test_recording.wav", seconds=5)

if audio_file:
    print("\nPlayback:")
    display(Audio(audio_file))

    result = whisper_model.transcribe(audio_file)
    print(f"\nYou said: {result['text']}")
    print(f"Language: {result['language']}")

---
# Part 3 -- Gemma: The LLM Brain (GIVEN)

We reuse **Gemma-4 E2B-it** from Day 18 Lab 2. This is the same model and `chat()` helper function you already know.

In [ ]:
# --- GIVEN: Load Gemma ---
from transformers import AutoProcessor, AutoModelForCausalLM, TextStreamer

GEMMA_ID = "google/gemma-4-E2B-it"

print(f"Loading {GEMMA_ID}...")
gemma_processor = AutoProcessor.from_pretrained(GEMMA_ID)
gemma_model = AutoModelForCausalLM.from_pretrained(
    GEMMA_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Gemma loaded ({sum(p.numel() for p in gemma_model.parameters()):,} parameters)")

In [ ]:
# --- GIVEN: Chat helper (same as Day 18 Lab 2) ---
def chat(messages, max_new_tokens=256, temperature=0.7, do_sample=True):
    """Send messages to Gemma and return the response."""
    text = gemma_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = gemma_processor(text=text, return_tensors="pt").to(gemma_model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = gemma_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
        )

    response = gemma_processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response.strip()


# Quick test
test_response = chat([{"role": "user", "content": "Say hello in one sentence."}])
print(f"Gemma: {test_response}")

---
# Part 4 -- Edge TTS: Text to Speech (GIVEN)

**Edge TTS** uses Microsoft's text-to-speech service (the same voices used in Microsoft Edge browser). It is free, requires no API key, and supports many languages and voices.

In [ ]:
# --- GIVEN: TTS helper ---
async def _tts_async(text, voice, output_file):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_file)

def text_to_speech(text, voice="en-US-AriaNeural", output_file="response.mp3"):
    """Convert text to speech and save as MP3."""
    asyncio.run(_tts_async(text, voice, output_file))
    return output_file


# Demo
tts_file = text_to_speech("Hello! I am your AI assistant. How can I help you today?")
print("TTS output:")
display(Audio(tts_file, autoplay=True))

In [ ]:
# --- GIVEN: Available voice options ---
VOICES = {
    "Aria (US Female)": "en-US-AriaNeural",
    "Guy (US Male)": "en-US-GuyNeural",
    "Sonia (British Female)": "en-GB-SoniaNeural",
    "Ryan (British Male)": "en-GB-RyanNeural",
    "Hamed (Arabic Male)": "ar-SA-HamedNeural",
    "Zariyah (Arabic Female)": "ar-SA-ZariyahNeural",
}

# Play a sample from each voice
for name, voice_id in VOICES.items():
    print(f"\n{name}:")
    tts_file = text_to_speech("The quick brown fox jumps over the lazy dog.", voice=voice_id, output_file=f"voice_{voice_id}.mp3")
    display(Audio(tts_file))

---
# Part 5 -- Full Voice Pipeline (GIVEN)

Now we chain everything together:
1. **Record** from your microphone
2. **Transcribe** with Whisper (speech -> text)
3. **Generate** response with Gemma (text -> text)
4. **Speak** the response with Edge TTS (text -> audio)
5. **Play** it back in the notebook

In [ ]:
# --- GIVEN: The full voice chat function ---
def voice_chat(system_prompt, voice="en-US-AriaNeural", history=None, record_seconds=5):
    """One turn of voice conversation.

    Records from mic, transcribes, gets LLM response, speaks it back.
    Returns the updated conversation history.
    """
    # Step 1: Record
    audio_file = record_audio("user_input.wav", seconds=record_seconds)
    if audio_file is None:
        return history

    # Step 2: Transcribe with Whisper
    result = whisper_model.transcribe(audio_file)
    user_text = result["text"].strip()
    print(f"\nYou said: {user_text}")

    if not user_text:
        print("(Could not understand audio. Try again.)")
        return history

    # Step 3: Build messages and get Gemma response
    if history is None:
        history = [{"role": "system", "content": system_prompt}]
    history.append({"role": "user", "content": user_text})

    response = chat(history, max_new_tokens=200)
    history.append({"role": "assistant", "content": response})
    print(f"Assistant: {response}")

    # Step 4: TTS
    tts_file = text_to_speech(response, voice=voice)

    # Step 5: Play back
    display(Audio(tts_file, autoplay=True))

    return history

print("Voice chat function ready.")

In [ ]:
# --- GIVEN: Try it! ---
# This will record 5 seconds of audio, transcribe it, get a response, and speak it back.
history = voice_chat(
    system_prompt="You are a friendly AI assistant. Keep your answers short -- 2 sentences max.",
    voice="en-US-AriaNeural",
)

In [ ]:
# --- GIVEN: Continue the conversation (multi-turn) ---
# Run this cell multiple times to keep talking!
history = voice_chat(
    system_prompt="You are a friendly AI assistant. Keep your answers short -- 2 sentences max.",
    voice="en-US-AriaNeural",
    history=history,  # pass the history to maintain context
)

---
# Part 6 -- Personalities

Remember from Day 18: **system prompts** control the model's personality. Now you get to combine a personality with a matching **voice**.

## Task 1: Create 3 Personalities

**TODO:** Define 3 personality configurations. Each should have:
- A **name**
- A **system prompt** that defines the personality
- A **voice** that matches the character (pick from the voices above)

Ideas:
- A pirate captain who only speaks in pirate slang
- A strict science professor who always asks follow-up questions
- A sports commentator who narrates everything dramatically
- A medieval knight who speaks in old English
- A detective solving a mystery

Fill in the dictionary below:

In [ ]:
personalities = {
    "personality_1": {
        "name": "",           # TODO: give it a name
        "system_prompt": "",   # TODO: describe the personality
        "voice": "",           # TODO: pick a voice ID
    },
    "personality_2": {
        "name": "",
        "system_prompt": "",
        "voice": "",
    },
    "personality_3": {
        "name": "",
        "system_prompt": "",
        "voice": "",
    },
}

## Task 2: Voice Conversation with a Personality

**TODO:** Pick your favorite personality from above and have a multi-turn voice conversation with it. Run the cell below multiple times to keep talking.

Change `chosen` to try different personalities.

In [ ]:
# TODO: Change "personality_1" to whichever you want to try
chosen = personalities["personality_1"]

print(f"Chatting with: {chosen['name']}")
print(f"Voice: {chosen['voice']}")
print(f"Personality: {chosen['system_prompt'][:80]}...")
print()

# Start a new conversation
personality_history = None

In [ ]:
# Run this cell each time you want to say something
personality_history = voice_chat(
    system_prompt=chosen["system_prompt"],
    voice=chosen["voice"],
    history=personality_history,
)

---
# Part 7 -- (Bonus) Arabic Voice Chatbot

## Task 3: Arabic Chatbot

**TODO:** Create an Arabic personality:
1. Write a system prompt in Arabic (e.g., "انت مساعد ذكي ودود. اجب دائما باللغة العربية بشكل مبسط ومختصر.")
2. Use an Arabic voice (`ar-SA-HamedNeural` or `ar-SA-ZariyahNeural`)
3. Speak in Arabic and see if the full pipeline works!

Whisper can transcribe Arabic, Gemma can respond in Arabic, and Edge TTS can speak Arabic.

In [ ]:
# Your code here


---
## Discussion

1. How well did Whisper transcribe your speech? Did it make any mistakes? What kinds of words were hardest?
2. Did the personality system prompts change how Gemma responded? Was the personality consistent across turns?
3. Which TTS voice sounded most natural to you? Did it match the personality well?
4. What was the biggest delay in the pipeline? Which step took the longest?
5. How well did the Arabic pipeline work compared to English? Any differences in quality?

---
## Wrap-Up

**What you learned:**
- **Whisper** converts speech to text in 99 languages with just 5 lines of code
- **Gemma** generates intelligent responses using system prompts and chat history (same as Day 18)
- **Edge TTS** converts text back to natural-sounding speech with multiple voices
- **Pipeline architecture**: each model is a specialist at one job -- chain them together to build something powerful
- The same pipeline works across languages: Arabic speech in, Arabic response out

**The pipeline you built:**

```
Microphone -> [Whisper: speech-to-text] -> [Gemma: understanding + response] -> [Edge TTS: text-to-speech] -> Speaker
```

This is the same architecture used by voice assistants like Siri, Alexa, and Google Assistant -- just with different (and larger) models at each step.